# 14 - Analisi di Firme Spettrali REALI (NIST WebBook)

Questo notebook è il seguito del **13**, ma con una differenza fondamentale:

> **Qui si usano dati spettroscopici REALI**, misurati in laboratorio,
> scaricati dal **NIST Chemistry WebBook** (spettri IR in formato JCAMP-DX).
> Nessuna gaussiana inventata, nessuno spettro "osservato" costruito con le stesse
> molecole che poi si vogliono ritrovare.

### Cosa cambia rispetto al notebook 13

| | Notebook 13 | Notebook 14 (questo) |
|---|---|---|
| Firme molecolari | Gaussiane sintetiche | **Spettri IR reali NIST** |
| Spettro "osservato" | Somma pesata delle stesse firme (circolare!) | **Miscela "nascosta"** che l'algoritmo deve scoprire |
| Identificazione | Confronto visivo | **Cross-correlazione quantitativa** con punteggio |
| Fonte dati | HITRAN (endpoint rotto → fallback finto) | NIST WebBook (funzionante, senza API key) |

### Limite onesto dei dati
Gli spettri IR del NIST coprono circa **2.5 – 20 µm** (infrarosso medio/termico,
la regione dello strumento **MIRI** di JWST). **Non** coprono il vicino-IR sotto
2.5 µm (regione NIRSpec usata nel nb 13): per quello servirebbe HITRAN con account.
Lavoriamo quindi nell'IR termico, dove i dati reali sono disponibili.

## 0. Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, re, time

np.random.seed(42)  # riproducibilità del rumore e della miscela nascosta
print('Librerie caricate')

## 1. Scaricamento dati reali dal NIST WebBook

Il NIST espone gli spettri IR in formato **JCAMP-DX** tramite un semplice URL,
identificando ogni molecola con il suo numero **CAS**.

⚠️ **Rate limiting**: il NIST blocca (HTTP 429) se le richieste sono troppo
ravvicinate. Per questo mettiamo una pausa (`time.sleep`) tra un download e
l'altro. Se una molecola viene bloccata, il codice riprova.

In [ ]:
# Numero CAS di ogni molecola sul NIST WebBook
NIST_CAS = {
    'H2O': 'C7732185',   # Acqua
    'CO2': 'C124389',    # Anidride carbonica
    'CH4': 'C74828',     # Metano
    'CO':  'C630080',    # Monossido di carbonio
    'NH3': 'C7664417',   # Ammoniaca
    'O3':  'C10028156',  # Ozono
    'SO2': 'C7446095',   # Biossido di zolfo
    'N2O': 'C10024972',  # Protossido d'azoto
}

LABELS = {
    'H2O':'H₂O (Acqua)','CO2':'CO₂','CH4':'CH₄ (Metano)','CO':'CO',
    'NH3':'NH₃ (Ammoniaca)','O3':'O₃ (Ozono)','SO2':'SO₂','N2O':'N₂O',
}
COLORS = {
    'H2O':'#2196F3','CO2':'#FF5722','CH4':'#4CAF50','CO':'#9C27B0',
    'NH3':'#FF9800','O3':'#00BCD4','SO2':'#E91E63','N2O':'#795548',
}

def download_nist_ir(cas, max_retries=3):
    """Scarica lo spettro IR JCAMP-DX di una molecola dal NIST WebBook.
    Ritorna il testo JCAMP oppure None."""
    url = 'https://webbook.nist.gov/cgi/cbook.cgi'
    params = {'JCAMP': cas, 'Index': 0, 'Type': 'IR'}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200 and 'INFRARED' in r.text:
                return r.text
            if r.status_code == 429:      # rate limited: aspetta di più
                time.sleep(3 * (attempt + 1))
                continue
        except Exception as e:
            print(f'    tentativo {attempt+1} fallito: {e}')
            time.sleep(2)
    return None

print('Funzione di download definita')

## 2. Parser JCAMP-DX

Il formato JCAMP codifica i dati come `(X++(Y..Y))`: ogni riga inizia con un
valore X (numero d'onda in cm⁻¹) seguito da più valori Y equispaziati di `DELTAX`.
I valori vanno moltiplicati per `XFACTOR`/`YFACTOR`.

Attenzione: alcuni spettri sono in **ASSORBANZA**, altri in **TRASMITTANZA**.
Li convertiamo tutti in "assorbanza relativa" per poterli confrontare.

In [ ]:
def parse_jcamp(text):
    """Decodifica un file JCAMP-DX. Ritorna (numero_donda_cm-1, assorbanza)."""
    hdr = {}
    data_lines = []
    in_data = False
    for line in text.splitlines():
        line = line.rstrip()
        if line.startswith('##'):
            if 'XYDATA' in line:
                in_data = True; continue
            if line.startswith('##END'):
                in_data = False; continue
            m = re.match(r'##\$?([^=]+)=(.*)', line)
            if m:
                hdr[m.group(1).strip()] = m.group(2).strip()
        elif in_data and line.strip():
            data_lines.append(line)

    xf = float(hdr.get('XFACTOR', 1.0))
    yf = float(hdr.get('YFACTOR', 1.0))
    dx = float(hdr['DELTAX'])
    yunits = hdr.get('YUNITS', 'ABSORBANCE').upper()

    xs, ys = [], []
    for dl in data_lines:
        toks = dl.split()
        x0 = float(toks[0]) * xf
        for i, t in enumerate(toks[1:]):
            xs.append(x0 + i * dx)
            ys.append(float(t) * yf)
    xs = np.array(xs); ys = np.array(ys)

    # Uniforma a "assorbanza": se è trasmittanza, converti
    if 'TRANSMITT' in yunits:
        ys = np.clip(ys, 0, None)
        ys = ys / ys.max() if ys.max() > 0 else ys      # normalizza a [0,1]
        ys = 1.0 - ys                                    # assorbanza = 1 - trasmittanza
    return xs, ys, yunits

print('Parser JCAMP definito')

## 3. Scarica tutte le molecole (dati reali)

In [ ]:
molecules = {}   # nome -> dict con wl (micron), assorbanza normalizzata

for name, cas in NIST_CAS.items():
    print(f'Scarico {LABELS[name]:20s} ...', end=' ')
    text = download_nist_ir(cas)
    if text is None:
        print('NON disponibile (skip)')
        continue
    wn, absb, yunits = parse_jcamp(text)      # wn in cm-1
    wl = 10000.0 / wn                          # cm-1 -> micron
    # ordina per lunghezza d'onda crescente
    order = np.argsort(wl)
    wl, absb = wl[order], absb[order]
    # normalizza l'assorbanza a [0,1] per confronti equi tra molecole
    if absb.max() > 0:
        absb = absb / absb.max()
    molecules[name] = {'wl': wl, 'abs': absb, 'yunits': yunits}
    print(f'OK  {len(wl)} punti  ({wl.min():.1f}-{wl.max():.1f} µm, {yunits})')
    time.sleep(1.5)   # <-- pausa anti rate-limit del NIST

print(f'\nMolecole reali caricate: {len(molecules)}/{len(NIST_CAS)}')
if len(molecules) == 0:
    print('ATTENZIONE: nessun dato scaricato. Controlla la connessione a Internet.')

## 4. Griglia comune

Ogni spettro NIST ha la sua griglia di punti. Per confrontarli li
**ricampioniamo** su una griglia comune di lunghezze d'onda con interpolazione.
Usiamo la regione realmente coperta dai dati: **2.5 – 20 µm**.

In [ ]:
WL_MIN, WL_MAX = 2.5, 20.0
wl_grid = np.linspace(WL_MIN, WL_MAX, 4000)

def resample(name):
    """Interpola lo spettro di una molecola sulla griglia comune."""
    m = molecules[name]
    # np.interp richiede x crescente; fuori range -> 0 (nessun dato)
    return np.interp(wl_grid, m['wl'], m['abs'], left=0.0, right=0.0)

grid_spectra = {name: resample(name) for name in molecules}
print(f'Spettri ricampionati su {len(wl_grid)} punti tra {WL_MIN} e {WL_MAX} µm')
print('Molecole:', ', '.join(grid_spectra.keys()))

## 5. Panoramica: tutte le firme reali

Prima di tutto, guardiamo che aspetto hanno gli spettri **veri**. Nota quanto
sono più ricchi e irregolari delle gaussiane pulite del notebook 13: questa è
la vera "impronta digitale" di ogni molecola.

In [ ]:
n = len(grid_spectra)
fig, axes = plt.subplots(n, 1, figsize=(13, 1.5*n), sharex=True)
if n == 1: axes = [axes]

for ax, (name, spec) in zip(axes, grid_spectra.items()):
    ax.fill_between(wl_grid, 0, spec, color=COLORS[name], alpha=0.85)
    ax.set_ylabel(LABELS[name], fontsize=9, rotation=0, ha='right', va='center')
    ax.set_yticks([]); ax.set_ylim(0, 1.05)
    ax.grid(True, axis='x', alpha=0.3)

axes[0].set_title('Firme spettrali IR REALI (NIST WebBook) — assorbanza normalizzata',
                  fontsize=13, fontweight='bold', pad=12)
axes[-1].set_xlabel('Lunghezza d\'onda (µm)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Il "pianeta misterioso" 🪐

Ora creiamo uno spettro **osservato** simulando un pianeta la cui atmosfera
contiene alcune molecole — **ma il codice qui sotto NON dice quali**. La miscela
è scelta casualmente e resta *nascosta*.

Questo elimina la circolarità del notebook 13: là lo spettro osservato era la
somma delle firme con pesi scritti a mano, quindi il "match" era garantito per
costruzione. Qui invece l'algoritmo dovrà **scoprire** cosa c'è, senza saperlo.

*(La soluzione è memorizzata in una variabile che riveleremo solo alla fine.)*

In [ ]:
available = list(grid_spectra.keys())

# Scegli casualmente 3-4 molecole presenti nel "pianeta", con abbondanze casuali
n_present = np.random.randint(3, min(5, len(available)) + 1)
hidden_names = list(np.random.choice(available, size=n_present, replace=False))
hidden_weights = {nm: round(np.random.uniform(0.3, 1.0), 2) for nm in hidden_names}

# Costruisci lo spettro osservato dalla miscela nascosta
observed = np.zeros_like(wl_grid)
for nm, w in hidden_weights.items():
    observed += w * grid_spectra[nm]

# Aggiungi rumore strumentale realistico
observed += np.random.normal(0, 0.03, len(wl_grid))
observed = np.clip(observed, 0, None)
if observed.max() > 0:
    observed /= observed.max()

print('Spettro del pianeta misterioso generato.')
print(f'(La miscela reale contiene {n_present} molecole — nascoste fino alla fine!)')

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(wl_grid, observed, color='#37474F', lw=1.0, label='Spettro osservato (pianeta X)')
plt.fill_between(wl_grid, 0, observed, color='#37474F', alpha=0.1)
plt.xlabel('Lunghezza d\'onda (µm)', fontsize=12)
plt.ylabel('Assorbanza normalizzata', fontsize=12)
plt.title('Spettro osservato dell\'atmosfera del "pianeta X" — quali molecole contiene?',
          fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.ylim(-0.05, 1.1)
plt.tight_layout(); plt.show()

## 7. Identificazione quantitativa

Come capire quali molecole ci sono? Combiniamo due approcci complementari:

1. **Correlazione di Pearson** (per molecola) — quanto la forma della firma
   "sale e scende insieme" allo spettro osservato. Indipendente dall'intensità.

2. **Fit ai minimi quadrati non-negativi (NNLS)** — chiediamo: *"con quali
   quantità (≥ 0) di ciascuna firma reale posso ricostruire lo spettro
   osservato?"*. È il cuore del metodo vero: invece di guardare una molecola
   alla volta, le mette **tutte in competizione** per spiegare l'osservato.
   Il coefficiente di ciascuna è la sua "abbondanza stimata".

Nessun peso imposto a mano: i coefficienti sono **ricavati dai dati reali**.

In [ ]:
from scipy.optimize import nnls

def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    denom = np.sqrt((a*a).sum() * (b*b).sum())
    return float((a*b).sum() / denom) if denom > 0 else 0.0

# --- Fit globale NNLS: osservato ≈ somma( coeff_i * firma_i ),  coeff_i >= 0 ---
names = list(grid_spectra.keys())
A = np.column_stack([grid_spectra[n] for n in names])   # matrice firme (punti x molecole)
coeffs, _ = nnls(A, observed)                            # coefficienti non-negativi
# normalizza i coefficienti a [0,1] per leggerli come "abbondanza stimata"
coeffs_norm = coeffs / coeffs.max() if coeffs.max() > 0 else coeffs

scores = []
for i, name in enumerate(names):
    scores.append({
        'Molecola': LABELS[name],
        'nome': name,
        'Correlazione': round(pearson(observed, grid_spectra[name]), 3),
        'Coeff. NNLS': round(float(coeffs_norm[i]), 3),
    })

df = pd.DataFrame(scores)
# Il punteggio finale è guidato dal fit globale (NNLS), che decide chi "vince"
df['Punteggio'] = df['Coeff. NNLS'].round(3)
df = df.sort_values('Punteggio', ascending=False).reset_index(drop=True)
df[['Molecola', 'Correlazione', 'Coeff. NNLS', 'Punteggio']]

## 8. Verdetto dell'algoritmo

Fissiamo una soglia sul punteggio: le molecole sopra soglia sono quelle che
l'algoritmo dichiara **presenti**. Poi le confrontiamo con la verità nascosta.

In [ ]:
THRESHOLD = 0.1   # una molecola è "presente" se il suo coeff. NNLS supera il 10% del massimo
detected = df[df['Punteggio'] >= THRESHOLD]['nome'].tolist()

print('=== VERDETTO DELL\'ALGORITMO ===')
print(f'Molecole rilevate (coeff. NNLS >= {THRESHOLD}):')
for nm in detected:
    print(f'   [+] {LABELS[nm]}')

print('\n=== VERITA (miscela nascosta) ===')
for nm, w in sorted(hidden_weights.items(), key=lambda x: -x[1]):
    print(f'   - {LABELS[nm]} (abbondanza reale {w})')

# Confronto
present = set(hidden_weights)
found = set(detected)
print('\n=== CONFRONTO ===')
print(f'  Corrette  (veri positivi):  {sorted(LABELS[n] for n in found & present)}')
print(f'  Mancate   (falsi negativi): {sorted(LABELS[n] for n in present - found)}')
print(f'  Sbagliate (falsi positivi): {sorted(LABELS[n] for n in found - present)}')

n_ok = len(found & present)
print(f'\n  Accuratezza: {n_ok}/{len(present)} molecole presenti individuate correttamente.')

## 9. Visualizzazione finale: osservato vs firme candidate

In [ ]:
top = df.head(6)
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
axes = axes.flatten()

for ax, (_, row) in zip(axes, top.iterrows()):
    name = row['nome']
    truth = ' [PRESENTE]' if name in hidden_weights else ' [assente]'
    ax.plot(wl_grid, observed, color='#555', lw=0.8, alpha=0.6, label='Osservato')
    ax.fill_between(wl_grid, 0, grid_spectra[name], color=COLORS[name],
                    alpha=0.5, label=LABELS[name])
    ax.set_title(f"{LABELS[name]}  |  coeff={row['Punteggio']}{truth}",
                 fontsize=11, fontweight='bold')
    ax.set_xlim(WL_MIN, WL_MAX); ax.set_ylim(0, 1.1)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlabel('µm', fontsize=9)

plt.suptitle('Confronto spettro osservato vs firme REALI (ranking per coeff. NNLS)',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout(); plt.show()

## 10. Cosa abbiamo imparato

1. **Dati reali ≠ dati inventati.** Le firme di questo notebook sono spettri IR
   misurati (NIST), non gaussiane. Sono irregolari, ricche, con bande sovrapposte:
   la realtà è più complicata del modello del nb 13.

2. **Niente circolarità.** Lo spettro osservato è una miscela *nascosta*:
   l'algoritmo la scopre da solo con la cross-correlazione. Se ci azzecca, è un
   risultato vero, non garantito per costruzione.

3. **I falsi positivi esistono.** Molecole con bande nella stessa regione (es.
   H₂O, CO₂, O₃ nell'IR termico) possono confondersi: guarda se compaiono falsi
   positivi/negativi nel verdetto. È *esattamente* il problema della
   spettroscopia esoplanetaria reale.

4. **Limiti onesti dei dati.** Copertura 2.5–20 µm (IR termico/MIRI). Per il
   vicino-IR (< 2.5 µm, NIRSpec) servirebbe HITRAN/HITEMP con account, o dati
   ExoMol. La risoluzione NIST è bassa: per l'analisi vera si usano modelli di
   trasferimento radiativo (petitRADTRANS, TauREx) su liste di righe ad alta
   risoluzione.

> **In breve:** questo notebook mostra il *metodo* corretto con *dati veri*.
> Riesegui la cella 6 (miscela nascosta) più volte: cambierà il pianeta, e
> vedrai l'algoritmo cavarsela — o sbagliare — in casi diversi.